# Comparison: Base GPT-2 vs DAPT Checkpoint

Loads a base GPT-2 model from OpenAI `.pkl` params, loads a DAPT model from a checkpoint created by `save_checkpoint()`, and compares token embedding vectors using cosine similarity; then looks at changes in next-token probabilities for representative texts.


## 1. Get directory paths

In [1]:
import os
import sys
import pickle
from pathlib import Path

import torch
import tiktoken

# CELL 1: Optional Google Drive mount (Colab only)
import os
from pathlib import Path

def in_colab() -> bool:
    return "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ or "google.colab" in str(getattr(__import__("sys"), "modules", {}))

IN_COLAB = in_colab()

if IN_COLAB:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        print("Mounted Google Drive at /content/drive")
    except Exception as e:
        print(f"Could not mount Google Drive: {e}")
else:
    print("Not running in Colab; skipping Drive mount.")

# CELL 2: Resolve PROJECT_ROOT (env override > Colab Drive default > search from cwd)
import os
import sys
from pathlib import Path

env_root = os.environ.get("LLM_PROJECT_ROOT", "").strip()
default_colab_root = Path("/content/drive/MyDrive/llm-from-scratch-drive")

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").exists():
            return p
    return start

if env_root:
    PROJECT_ROOT = Path(env_root).expanduser().resolve()
elif default_colab_root.exists():
    PROJECT_ROOT = default_colab_root.resolve()
else:
    PROJECT_ROOT = find_repo_root(Path.cwd().resolve())

print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC exists:", (PROJECT_ROOT / "src").exists())

# CELL 4: Paths + sys.path + project imports
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"

# Create output dir if missing (safe)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("SRC_DIR:", SRC_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

# Add src to import path (notebook-friendly)
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Now your imports work locally or in Colab
from llm_from_scratch.configs import gpt2small_config
from llm_from_scratch.training import training_utils
from llm_from_scratch.models import gpt2
from llm_from_scratch.dataloader import dataloader

Mounted at /content/drive
Mounted Google Drive at /content/drive
CWD: /content
PROJECT_ROOT: /content/drive/MyDrive/llm-from-scratch-drive
SRC exists: True
SRC_DIR: /content/drive/MyDrive/llm-from-scratch-drive/src
DATA_DIR: /content/drive/MyDrive/llm-from-scratch-drive/data
OUTPUT_DIR: /content/drive/MyDrive/llm-from-scratch-drive/output


## 2. Import modules and establish tokenizer

In [2]:
from llm_from_scratch.models import gpt2
from llm_from_scratch.training import training_utils
from llm_from_scratch.configs import gpt2small_config
from llm_from_scratch.analysis import token_analysis as ta
from llm_from_scratch.analysis import weight_diff as wd

tokenizer = tiktoken.get_encoding("gpt2")
ta.tokenizer = tokenizer


## DEVICE/FILE PATH/FILE EXISTENCE  
Note that if DAPT file is not found, probably is in a subdirectory of /output:  
To fix:  
1. move to ./llm-from-scratch/data  
2. The file will also need to be renamed to "TEST_abstracts_epoch_lastsave_step_lastsave.pth"  

In [3]:
# Device selection: prefer CUDA if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Default file paths (edit if needed)
BASE_PARAMS_PATH = DATA_DIR / "gpt2_openai_params_124M.pkl"
DAPT_CKPT_PATH = DATA_DIR / "TEST_abstracts_epoch_lastsave_step_lastsave.pth"

print("BASE_PARAMS_PATH exists:", BASE_PARAMS_PATH.exists(), BASE_PARAMS_PATH)
print("DAPT_CKPT_PATH exists:", DAPT_CKPT_PATH.exists(), DAPT_CKPT_PATH)

if not BASE_PARAMS_PATH.exists():
    raise FileNotFoundError(f"Base params file not found: {BASE_PARAMS_PATH}")
if not DAPT_CKPT_PATH.exists():
    print("***File below is not found. Check in the output directory and subdirectory. Move to /data and rename to TEST_abstracts_epoch_lastsave_step_lastsave.pth")
    raise FileNotFoundError(f"DAPT checkpoint file not found: {DAPT_CKPT_PATH}")


Using device: cpu
BASE_PARAMS_PATH exists: True /content/drive/MyDrive/llm-from-scratch-drive/data/gpt2_openai_params_124M.pkl
DAPT_CKPT_PATH exists: True /content/drive/MyDrive/llm-from-scratch-drive/data/TEST_abstracts_epoch_lastsave_step_lastsave.pth


## 3. Load base model from OpenAI .pkl params

In [4]:

base_cfg = dict(gpt2small_config.GPT_CONFIG_124M_OPENAI)
base_model = gpt2.setup_model(base_cfg).to(device)

with open(BASE_PARAMS_PATH, "rb") as f:
    base_params = pickle.load(f)

training_utils.load_weights_into_gpt(base_model, base_params)
base_model.out_head.weight = base_model.tok_emb.weight
if base_model.out_head.weight is not base_model.tok_emb.weight:
    raise ValueError("base_model lost weight tying between tok_emb and out_head")
base_model.eval()
print("Loaded base model from:", BASE_PARAMS_PATH)
print("Base tok_emb shape:", tuple(base_model.tok_emb.weight.shape))


Loaded base model from: /content/drive/MyDrive/llm-from-scratch-drive/data/gpt2_openai_params_124M.pkl
Base tok_emb shape: (50257, 768)


In [5]:
# Inspect number of tensors total and in certain layers
base_named_params = list(base_model.named_parameters())
print("Number of named-parameter tensors in base_model:", len(base_named_params))

for block_idx in [0, 11]:
    block_prefix = f"trf_blocks.{block_idx}."
    block_param_names = [
        name for name, _ in base_named_params if name.startswith(block_prefix)
    ]
    print(f"\nNamed-parameter tensors in trf_blocks[{block_idx}] ({len(block_param_names)} total):")
    for name in block_param_names:
        print(name)


Number of named-parameter tensors in base_model: 196

Named-parameter tensors in trf_blocks[0] (16 total):
trf_blocks.0.att.W_query.weight
trf_blocks.0.att.W_query.bias
trf_blocks.0.att.W_key.weight
trf_blocks.0.att.W_key.bias
trf_blocks.0.att.W_value.weight
trf_blocks.0.att.W_value.bias
trf_blocks.0.att.out_proj.weight
trf_blocks.0.att.out_proj.bias
trf_blocks.0.ff.layers.0.weight
trf_blocks.0.ff.layers.0.bias
trf_blocks.0.ff.layers.2.weight
trf_blocks.0.ff.layers.2.bias
trf_blocks.0.norm1.scale
trf_blocks.0.norm1.shift
trf_blocks.0.norm2.scale
trf_blocks.0.norm2.shift

Named-parameter tensors in trf_blocks[11] (16 total):
trf_blocks.11.att.W_query.weight
trf_blocks.11.att.W_query.bias
trf_blocks.11.att.W_key.weight
trf_blocks.11.att.W_key.bias
trf_blocks.11.att.W_value.weight
trf_blocks.11.att.W_value.bias
trf_blocks.11.att.out_proj.weight
trf_blocks.11.att.out_proj.bias
trf_blocks.11.ff.layers.0.weight
trf_blocks.11.ff.layers.0.bias
trf_blocks.11.ff.layers.2.weight
trf_blocks.11.ff.

## 4. Load DAPT model from .pth checkpoint produced by save_checkpoint()
Note DAPT model was made by additional training with biomedical abstracts; see README.md

In [6]:

checkpoint = torch.load(DAPT_CKPT_PATH, map_location=device)

if "model_state_dict" not in checkpoint:
    raise KeyError("Checkpoint does not contain 'model_state_dict'.")

if "run_config" in checkpoint and "model_config" in checkpoint["run_config"]:
    dapt_model_cfg = checkpoint["run_config"]["model_config"]
else:
    # Fallback if run_config is missing
    dapt_model_cfg = base_cfg

dapt_model = gpt2.setup_model(dapt_model_cfg).to(device)
dapt_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
dapt_model.out_head.weight = dapt_model.tok_emb.weight
if dapt_model.out_head.weight is not dapt_model.tok_emb.weight:
    raise ValueError("dapt_model lost weight tying between tok_emb and out_head")
dapt_model.eval()

print("Loaded DAPT model from:", DAPT_CKPT_PATH)
print("DAPT tok_emb shape:", tuple(dapt_model.tok_emb.weight.shape))


Loaded DAPT model from: /content/drive/MyDrive/llm-from-scratch-drive/data/TEST_abstracts_epoch_lastsave_step_lastsave.pth
DAPT tok_emb shape: (50257, 768)


## 5. Model Difference Analysis Sections: Comparing the base (publically available weights) to the DAPT (additional training on biomedical abstracts) model


### 5a. Look at changes at the token embedding layer (+ output layer, as there is weight tying in this model)

In [7]:
# Extract token embedding matrices (base vs DAPT)
embed_initial = base_model.tok_emb.weight.detach().float().cpu().clone()
embed_after = dapt_model.tok_emb.weight.detach().float().cpu().clone()

print("shape_before:", tuple(embed_initial.shape))
print("shape_after:", tuple(embed_after.shape))


shape_before: (50257, 768)
shape_after: (50257, 768)


#### 5a.1 Do pairs of tokens that are closely associated in the biomedical text become closer in embedding in the DAPT model?

In [8]:
# Token-pair cosine similarity checks
import pandas as pd

words_dict = {
    "sp_HER2": tokenizer.encode(" HER2"),
    "HER_sp_2": tokenizer.encode("HER 2"),
    "sp_EGFR": tokenizer.encode(" EGFR"),
    "EG_sp_FR": tokenizer.encode("EG FR"),
    "ERBB2_only_BB2": tokenizer.encode("BB2"),
    "ERBB2_only_spERBB": tokenizer.encode(" ERBB"),
    "sp_kinase": tokenizer.encode(" kinase"),
    "cat vs dog": [tokenizer.encode(" cat")[0], tokenizer.encode(" dog")[0]],
}

resdf = pd.DataFrame(columns=[
    "word", "tokenid1", "token1", "tokenid2", "token2", "cos_sim_openai", "cos_sim_aftertrain", "ratio_afterVSbefore"
])

for word, (tokenid1, tokenid2) in words_dict.items():
    cos_sim_openai = ta.compute_cosine_similarity(tokenid1, tokenid2, embed_initial)
    cos_sim_aftertrain = ta.compute_cosine_similarity(tokenid1, tokenid2, embed_after)
    resdf.loc[len(resdf)] = {
        "word": word,
        "tokenid1": tokenid1,
        "token1": repr(tokenizer.decode([tokenid1])),
        "tokenid2": tokenid2,
        "token2": repr(tokenizer.decode([tokenid2])),
        "cos_sim_openai": cos_sim_openai,
        "cos_sim_aftertrain": cos_sim_aftertrain,
        "ratio_afterVSbefore": cos_sim_aftertrain / cos_sim_openai,
    }

print(resdf.to_string(index=False))


             word  tokenid1 token1  tokenid2 token2  cos_sim_openai  cos_sim_aftertrain  ratio_afterVSbefore
          sp_HER2     24906 ' HER'        17    '2'        0.259792            0.265321             1.021285
         HER_sp_2     16879  'HER'       362   ' 2'        0.171015            0.172065             1.006138
          sp_EGFR     41513  ' EG'     10913   'FR'        0.256762            0.243429             0.948070
         EG_sp_FR      7156   'EG'      8782  ' FR'        0.266662            0.230426             0.864115
   ERBB2_only_BB2     15199   'BB'        17    '2'        0.275225            0.270166             0.981620
ERBB2_only_spERBB     13793  ' ER'     15199   'BB'        0.255247            0.263833             1.033637
        sp_kinase     18967 ' kin'       589  'ase'        0.270267            0.271449             1.004375
       cat vs dog      3797 ' cat'      3290 ' dog'        0.549790            0.528729             0.961694


**RESULT**: These token-pair embedding checks show only small local shifts. That is useful as a negative result: simple biomedical token-pair proximity is not where the main DAPT effect shows up most clearly. This is consistent with the report's conclusion that the strongest changes are distributed across the model and are better captured by prompt behavior and weight-transplant results than by a few embedding-pair comparisons.

#### 5a.2 Do token embeddings stay basically the same across base model vs DAPT model?

In [9]:
# Extract token embedding matrices and compute cosine similarity per token for base vs DAPT
base_tok_emb = base_model.tok_emb.weight.detach().float().cpu()
dapt_tok_emb = dapt_model.tok_emb.weight.detach().float().cpu()

if base_tok_emb.shape != dapt_tok_emb.shape:
    raise ValueError(
        f"Embedding shape mismatch: base={tuple(base_tok_emb.shape)} vs dapt={tuple(dapt_tok_emb.shape)}"
    )

cos_scores = ta.cosine_similarity_per_token(base_tok_emb, dapt_tok_emb)
print("cos_scores shape:", tuple(cos_scores.shape))
print("cosine min/max/mean:", cos_scores.min().item(), cos_scores.max().item(), cos_scores.mean().item())


cos_scores shape: (50257,)
cosine min/max/mean: 0.8702036738395691 0.999301552772522 0.9683943390846252


In [10]:
# 4) Rank top 40 most changed and least changed tokens
TOP_K = 40

most_changed_df = ta.rank_tokens_by_cosine_similarity(
    cosine_scores=cos_scores,
    tokenizer=tokenizer,
    k=TOP_K,
    mode="most_dissimilar",
)

least_changed_df = ta.rank_tokens_by_cosine_similarity(
    cosine_scores=cos_scores,
    tokenizer=tokenizer,
    k=TOP_K,
    mode="most_similar",
)

print("Top 40 MOST changed tokens (lowest cosine):")
print(most_changed_df.to_string(index=False))

print()
print("Top 40 LEAST changed tokens (highest cosine):")
print(least_changed_df.to_string(index=False))


Top 40 MOST changed tokens (lowest cosine):
 tokenid        token  cosine_similarity
     921       ' You'           0.870204
    2130   ' someone'           0.879040
    5137   ' putting'           0.879239
    2094       ' Don'           0.882754
    6209 ' basically'           0.884582
    4705      ' Matt'           0.885449
    1532         'If'           0.886136
    2495    ' pretty'           0.886501
    1639        'You'           0.887201
     534      ' your'           0.887860
    7360 ' literally'           0.889315
    1223 ' something'           0.890046
     345       ' you'           0.890096
    1793       ' God'           0.893600
    3977   ' William'           0.893915
    9775 ' seemingly'           0.894630
    1770        ' Mr'           0.894895
    6079  ' bringing'           0.895265
    2396         'So'           0.895715
    4995      ' Mike'           0.895818
    7214      ' Take'           0.895923
    4302 ' Christian'           0.896345
    3932     

**RESULT**: The base-vs-DAPT embedding matrix did change, but the most changed individual tokens are not obviously concentrated on biomedical vocabulary. This weakens a narrow interpretation in which DAPT mainly rewired a small set of domain-specific token embeddings, and instead supports the broader report conclusion that the adaptation is distributed.

#### 5a.3 Do next token probabilities shift?

In [11]:

GPROMPTS = [
    " dogs and",
    "Many families like keeping animals in their homes - we call these pets. Among the most popular pets are dogs and",
    " HER",
    "For breast cancer, trastuzumab, neratinib, and tucatinib are used to treat HER",
    "We visited Greek temples, and saw the names of various Greek gods from Greek mythology, including ZEUS, POSEIDON, APOLLO, HADES and finally one for the brave HER",
    " ER",
    "Although HER2 amplification is usually considered in the context of breast cancer, it is clear that it can be amplified in other types of cancer also. In colorectal cancer, a subset of patients are found to have HER2 amplification as indicated by copy number increases in ER",
    "We visited the hospital, which is famous for all the gunshot wounds that come into the emergency room. We were told that our friend was being treated by a doctor in the ER",
    ]

TOP_K_NEXT = 10

base_context_size = int(base_cfg["context_length"])
dapt_context_size = int(dapt_model_cfg["context_length"])

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)

for idx, prompt in enumerate(GPROMPTS, start=1):
    base_df = ta.top_next_tokens(base_model, prompt, base_context_size, top_k=TOP_K_NEXT)
    dapt_df = ta.top_next_tokens(dapt_model, prompt, dapt_context_size, top_k=TOP_K_NEXT)

    comparison_df = pd.concat(
        [base_df.add_prefix("base_"), dapt_df.add_prefix("dapt_")],
        axis=1,
    )

    print(f"PROMPT{idx:02d}:", prompt)
    print()
    print("Base vs DAPT top next tokens:")
    print(comparison_df)
    print("\n" + "-" * 80 + "\n")

# check ERBB2 predicted prompt specifically
print("BB predicted next token analysis")
print(GPROMPTS[6])
BBprob_base = ta.next_token_probability(base_model, GPROMPTS[6], "BB", base_context_size)
BBprob_DAPT = ta.next_token_probability(dapt_model, GPROMPTS[6], "BB", dapt_context_size)
print(f"Base P('BB'): {BBprob_base:.6f}")
print(f"DAPT P('BB'): {BBprob_DAPT:.6f}")


PROMPT01:  dogs and

Base vs DAPT top next tokens:
   base_rank  base_tokenid   base_token  base_probability  dapt_rank  dapt_tokenid   dapt_token  dapt_probability
0          1           584     ' other'          0.042839          1           262       ' the'          0.053074
1          2           511     ' their'          0.037026          2           511     ' their'          0.039474
2          3          6844      ' dogs'          0.036724          3           284        ' to'          0.022300
3          4         11875      ' cats'          0.027796          4           257         ' a'          0.016375
4          5           257         ' a'          0.024118          5          3871  ' patients'          0.015572
5          6           262       ' the'          0.023013          6          4890    ' cancer'          0.014867
6          7          1751  ' children'          0.014539          7         44678    ' metast'          0.014780
7          8         14260    ' horse

**RESULT**: The prompt comparisons are the strongest behavioral evidence in this notebook. Some control prompts change only modestly, which shows context still matters, but the biomedical prompts show large context-sensitive shifts, especially the ERBB2-style completion where the DAPT model strongly favors 'BB' and the base model does not. The emergency-room control also shows that DAPT does not blindly force the same continuation in every ' ER' context. Taken together, these results support the report's main conclusion: DAPT changed model behavior in a distributed way. Embedding changes are part of that story, but they are not sufficient on their own to explain the behavioral shift or the validation-loss advantage.  

### 5b. Which parameters and model blocks changed most after DAPT?

In [12]:
num_to_print=200
param_df = wd.compare_models(base_model, dapt_model, max_changed_tensors=num_to_print)
block_df = wd.summarize_by_gpt2_block(param_df)
block_proper_df = wd.summarize_by_block_proper(param_df, base_model, dapt_model)
submodule_df = wd.summarize_by_submodule(param_df)

print("Top changed individual tensors:")
print(param_df.head(num_to_print).to_string(index=False))

print("\nChanged blocks (simple summary):")
print(block_df.to_string(index=False))

print("\nChanged blocks (proper aggregated relative L2):")
print(block_proper_df.to_string(index=False))

print("\nChanged submodules:")
print(submodule_df.to_string(index=False))


Top changed individual tensors:
                             name    numel   rel_l2  delta_norm  mean_abs_delta  max_abs_delta  cosine_similarity  base_norm  dapt_norm
        trf_blocks.11.norm2.shift      768 0.260089    0.289773        0.008229       0.041500           0.966056   1.114131   1.109945
                   tok_emb.weight 38597376 0.248346  221.147263        0.027622       0.326320           0.973790 890.481873 877.361938
         trf_blocks.0.norm1.shift      768 0.246904    0.248933        0.006860       0.049154           0.969894   1.008217   0.936842
        trf_blocks.10.norm2.shift      768 0.167272    0.229862        0.006481       0.043087           0.985969   1.374178   1.340163
         trf_blocks.4.norm2.shift      768 0.145634    0.108501        0.003101       0.019291           0.989391   0.745021   0.744715
  trf_blocks.0.att.W_value.weight   589824 0.141011    6.293947        0.006421       0.043385           0.990113  44.634369  43.567841
         trf_blo

## 6. Base model token embedding replaced with DAPT token embedding  

In [13]:
base_embed_plusothers_dapt_model = gpt2.setup_model(base_cfg).to(device)
training_utils.load_weights_into_gpt(base_embed_plusothers_dapt_model, base_params)
base_embed_plusothers_dapt_model.out_head.weight = base_embed_plusothers_dapt_model.tok_emb.weight

if base_embed_plusothers_dapt_model.tok_emb.weight.shape != dapt_model.tok_emb.weight.shape:
    raise ValueError(
        f"tok_emb shape mismatch: {tuple(base_embed_plusothers_dapt_model.tok_emb.weight.shape)} vs {tuple(dapt_model.tok_emb.weight.shape)}"
    )

if base_embed_plusothers_dapt_model.pos_emb.weight.shape != dapt_model.pos_emb.weight.shape:
    raise ValueError(
        f"pos_emb shape mismatch: {tuple(base_embed_plusothers_dapt_model.pos_emb.weight.shape)} vs {tuple(dapt_model.pos_emb.weight.shape)}"
    )

with torch.no_grad():
    base_embed_plusothers_dapt_model.tok_emb.weight.copy_(dapt_model.tok_emb.weight)
    base_embed_plusothers_dapt_model.pos_emb.weight.copy_(dapt_model.pos_emb.weight)
    base_embed_plusothers_dapt_model.trf_blocks[0].norm1.shift.copy_(dapt_model.trf_blocks[0].norm1.shift)
    base_embed_plusothers_dapt_model.trf_blocks[4].norm2.shift.copy_(dapt_model.trf_blocks[4].norm2.shift)
    base_embed_plusothers_dapt_model.trf_blocks[9].norm2.shift.copy_(dapt_model.trf_blocks[9].norm2.shift)
    base_embed_plusothers_dapt_model.trf_blocks[10].norm2.shift.copy_(dapt_model.trf_blocks[10].norm2.shift)
    base_embed_plusothers_dapt_model.trf_blocks[11].norm2.shift.copy_(dapt_model.trf_blocks[11].norm2.shift)

if base_embed_plusothers_dapt_model.out_head.weight is not base_embed_plusothers_dapt_model.tok_emb.weight:
    raise ValueError("base_embed_plusothers_dapt_model lost weight tying between tok_emb and out_head")

base_embed_plusothers_dapt_model.eval()
print("Loaded base_embed_plusothers_dapt_model as a duplicate of base_model")
print("Replaced base_embed_plusothers_dapt_model tok_emb with DAPT tok_emb")
print("Replaced base_embed_plusothers_dapt_model pos_emb with DAPT pos_emb")
print("Replaced base_embed_plusothers_dapt_model trf_blocks.0.norm1.shift with DAPT value")
print("Replaced base_embed_plusothers_dapt_model trf_blocks.4.norm2.shift with DAPT value")
print("Replaced base_embed_plusothers_dapt_model trf_blocks.9.norm2.shift with DAPT value")
print("Replaced base_embed_plusothers_dapt_model trf_blocks.10.norm2.shift with DAPT value")
print("Replaced base_embed_plusothers_dapt_model trf_blocks.11.norm2.shift with DAPT value")
print("base_embed_plusothers_dapt_model tok_emb shape:", tuple(base_embed_plusothers_dapt_model.tok_emb.weight.shape))

# create base_embed_pos_trf0_11_model
base_embed_pos_trf0_11_model = gpt2.setup_model(base_cfg).to(device)
training_utils.load_weights_into_gpt(base_embed_pos_trf0_11_model, base_params)
base_embed_pos_trf0_11_model.out_head.weight = base_embed_pos_trf0_11_model.tok_emb.weight

base_embed_pos_trf0_11_sd = base_embed_pos_trf0_11_model.state_dict()
dapt_sd = dapt_model.state_dict()
prefixes_to_replace = ["tok_emb.", "pos_emb.", "trf_blocks.0.", "trf_blocks.11."]
replaced_tensor_names = []

with torch.no_grad():
    for tensor_name, target_tensor in base_embed_pos_trf0_11_sd.items():
        if not any(tensor_name.startswith(prefix) for prefix in prefixes_to_replace):
            continue
        if tensor_name not in dapt_sd:
            raise KeyError(f"{tensor_name} missing from dapt_model state_dict")
        source_tensor = dapt_sd[tensor_name]
        if target_tensor.shape != source_tensor.shape:
            raise ValueError(
                f"shape mismatch for {tensor_name}: {tuple(target_tensor.shape)} vs {tuple(source_tensor.shape)}"
            )
        target_tensor.copy_(source_tensor)
        replaced_tensor_names.append(tensor_name)

if base_embed_pos_trf0_11_model.out_head.weight is not base_embed_pos_trf0_11_model.tok_emb.weight:
    raise ValueError("base_embed_pos_trf0_11_model lost weight tying between tok_emb and out_head")

base_embed_pos_trf0_11_model.eval()
print("Loaded base_embed_pos_trf0_11_model as a duplicate of base_model")
print("Replaced tok_emb, pos_emb, trf_blocks.0.*, and trf_blocks.11.* with DAPT values")
print("Number of tensors replaced in base_embed_pos_trf0_11_model:", len(replaced_tensor_names))
print("base_embed_pos_trf0_11_model tok_emb shape:", tuple(base_embed_pos_trf0_11_model.tok_emb.weight.shape))

# create base_embed_pos_trf5_6_model
base_embed_pos_trf5_6_model = gpt2.setup_model(base_cfg).to(device)
training_utils.load_weights_into_gpt(base_embed_pos_trf5_6_model, base_params)
base_embed_pos_trf5_6_model.out_head.weight = base_embed_pos_trf5_6_model.tok_emb.weight

base_embed_pos_trf5_6_sd = base_embed_pos_trf5_6_model.state_dict()
prefixes_to_replace = ["tok_emb.", "pos_emb.", "trf_blocks.5.", "trf_blocks.6."]
replaced_tensor_names = []

with torch.no_grad():
    for tensor_name, target_tensor in base_embed_pos_trf5_6_sd.items():
        if not any(tensor_name.startswith(prefix) for prefix in prefixes_to_replace):
            continue
        if tensor_name not in dapt_sd:
            raise KeyError(f"{tensor_name} missing from dapt_model state_dict")
        source_tensor = dapt_sd[tensor_name]
        if target_tensor.shape != source_tensor.shape:
            raise ValueError(
                f"shape mismatch for {tensor_name}: {tuple(target_tensor.shape)} vs {tuple(source_tensor.shape)}"
            )
        target_tensor.copy_(source_tensor)
        replaced_tensor_names.append(tensor_name)

if base_embed_pos_trf5_6_model.out_head.weight is not base_embed_pos_trf5_6_model.tok_emb.weight:
    raise ValueError("base_embed_pos_trf5_6_model lost weight tying between tok_emb and out_head")

base_embed_pos_trf5_6_model.eval()
print("Loaded base_embed_pos_trf5_6_model as a duplicate of base_model")
print("Replaced tok_emb, pos_emb, trf_blocks.5.*, and trf_blocks.6.* with DAPT values")
print("Number of tensors replaced in base_embed_pos_trf5_6_model:", len(replaced_tensor_names))
print("base_embed_pos_trf5_6_model tok_emb shape:", tuple(base_embed_pos_trf5_6_model.tok_emb.weight.shape))

# create base_embed_dapt model
base_embed_dapt_model = gpt2.setup_model(base_cfg).to(device)
training_utils.load_weights_into_gpt(base_embed_dapt_model, base_params)
base_embed_dapt_model.out_head.weight = base_embed_dapt_model.tok_emb.weight

if base_embed_dapt_model.tok_emb.weight.shape != dapt_model.tok_emb.weight.shape:
    raise ValueError(
        f"tok_emb shape mismatch: {tuple(base_embed_dapt_model.tok_emb.weight.shape)} vs {tuple(dapt_model.tok_emb.weight.shape)}"
    )

with torch.no_grad():
    base_embed_dapt_model.tok_emb.weight.copy_(dapt_model.tok_emb.weight)

if base_embed_dapt_model.out_head.weight is not base_embed_dapt_model.tok_emb.weight:
    raise ValueError("base_embed_dapt_model lost weight tying between tok_emb and out_head")

base_embed_dapt_model.eval()
print("Loaded base_embed_dapt_model as a duplicate of base_model")
print("Replaced base_embed_dapt_model tok_emb with DAPT tok_emb")
print("base_embed_dapt_model tok_emb shape:", tuple(base_embed_dapt_model.tok_emb.weight.shape))

# create dapt_embed_base model
dapt_embed_base_model = gpt2.setup_model(dapt_model_cfg).to(device)
dapt_embed_base_model.load_state_dict(dapt_model.state_dict(), strict=True)

if dapt_embed_base_model.tok_emb.weight.shape != base_model.tok_emb.weight.shape:
    raise ValueError(
        f"tok_emb shape mismatch: {tuple(dapt_embed_base_model.tok_emb.weight.shape)} vs {tuple(base_model.tok_emb.weight.shape)}"
    )

with torch.no_grad():
    dapt_embed_base_model.tok_emb.weight.copy_(base_model.tok_emb.weight)

if dapt_embed_base_model.out_head.weight is not dapt_embed_base_model.tok_emb.weight:
    raise ValueError("dapt_embed_base_model lost weight tying between tok_emb and out_head")

dapt_embed_base_model.eval()
print("Loaded dapt_embed_base_model as a duplicate of dapt_model")
print("Replaced dapt_embed_base_model tok_emb with base tok_emb")
print("dapt_embed_base_model tok_emb shape:", tuple(dapt_embed_base_model.tok_emb.weight.shape))


Loaded base_embed_plusothers_dapt_model as a duplicate of base_model
Replaced base_embed_plusothers_dapt_model tok_emb with DAPT tok_emb
Replaced base_embed_plusothers_dapt_model pos_emb with DAPT pos_emb
Replaced base_embed_plusothers_dapt_model trf_blocks.0.norm1.shift with DAPT value
Replaced base_embed_plusothers_dapt_model trf_blocks.4.norm2.shift with DAPT value
Replaced base_embed_plusothers_dapt_model trf_blocks.9.norm2.shift with DAPT value
Replaced base_embed_plusothers_dapt_model trf_blocks.10.norm2.shift with DAPT value
Replaced base_embed_plusothers_dapt_model trf_blocks.11.norm2.shift with DAPT value
base_embed_plusothers_dapt_model tok_emb shape: (50257, 768)
Loaded base_embed_pos_trf0_11_model as a duplicate of base_model
Replaced tok_emb, pos_emb, trf_blocks.0.*, and trf_blocks.11.* with DAPT values
Number of tensors replaced in base_embed_pos_trf0_11_model: 36
base_embed_pos_trf0_11_model tok_emb shape: (50257, 768)
Loaded base_embed_pos_trf5_6_model as a duplicate of

In [14]:
TOP_K_NEXT = 30

base_alt_context_size = int(base_cfg["context_length"])
dapt_context_size = int(dapt_model_cfg["context_length"])

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)

for idx, prompt in enumerate(GPROMPTS, start=1):
    base_alt_df = ta.top_next_tokens(base_embed_plusothers_dapt_model, prompt, base_alt_context_size, top_k=TOP_K_NEXT)
    dapt_df = ta.top_next_tokens(dapt_model, prompt, dapt_context_size, top_k=TOP_K_NEXT)

    comparison_df = pd.concat(
        [base_alt_df.add_prefix("base_alt_"), dapt_df.add_prefix("dapt_")],
        axis=1,
    )

    print(f"PROMPT{idx:02d}:", prompt)
    print()
    print("Base_alt vs DAPT top next tokens:")
    print(comparison_df)
    print("\n" + "-" * 80 + "\n")

# check ERBB2 predicted prompt specifically
print("BB predicted next token analysis")
print(GPROMPTS[6])
BBprob_base_alt = ta.next_token_probability(base_embed_plusothers_dapt_model, GPROMPTS[6], "BB", base_alt_context_size)
BBprob_DAPT = ta.next_token_probability(dapt_model, GPROMPTS[6], "BB", dapt_context_size)
print(f"Base_alt P('BB'): {BBprob_base_alt:.6f}")
print(f"DAPT P('BB'): {BBprob_DAPT:.6f}")


PROMPT01:  dogs and

Base_alt vs DAPT top next tokens:
    base_alt_rank  base_alt_tokenid  base_alt_token  base_alt_probability  dapt_rank  dapt_tokenid   dapt_token  dapt_probability
0               1               257            ' a'              0.050939          1           262       ' the'          0.053074
1               2               262          ' the'              0.048072          2           511     ' their'          0.039474
2               3               511        ' their'              0.043975          3           284        ' to'          0.022300
3               4              3114       ' looked'              0.029420          4           257         ' a'          0.016375
4               5               584        ' other'              0.027558          5          3871  ' patients'          0.015572
5               6               285            ' m'              0.021920          6          4890    ' cancer'          0.014867
6               7              1402

### dapt_embed_base_model vs base_model prompt-based evaluation

In [15]:
TOP_K_NEXT = 10

# context sizes are the same for all the models, hence this is not a big deal
base_context_size = int(base_cfg["context_length"])
dapt_context_size = base_context_size

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)

for idx, prompt in enumerate(GPROMPTS, start=1):
    base_df = ta.top_next_tokens(base_model, prompt, base_alt_context_size, top_k=TOP_K_NEXT)
    dapt_alt_df = ta.top_next_tokens(dapt_embed_base_model, prompt, dapt_context_size, top_k=TOP_K_NEXT)

    comparison_df = pd.concat(
        [base_df.add_prefix("base_"), dapt_alt_df.add_prefix("dapt_alt_")],
        axis=1,
    )

    print(f"PROMPT{idx:02d}:", prompt)
    print()
    print("Base vs DAPT_alt top next tokens:")
    print(comparison_df)
    print("\n" + "-" * 80 + "\n")

# check ERBB2 predicted prompt specifically
print("BB predicted next token analysis")
print(GPROMPTS[6])
BBprob_base = ta.next_token_probability(base_model, GPROMPTS[6], "BB", base_context_size)
BBprob_DAPT_alt = ta.next_token_probability(dapt_embed_base_model, GPROMPTS[6], "BB", dapt_context_size)
print(f"Base P('BB'): {BBprob_base:.6f}")
print(f"DAPT_alt P('BB'): {BBprob_DAPT_alt:.6f}")


PROMPT01:  dogs and

Base vs DAPT_alt top next tokens:
   base_rank  base_tokenid   base_token  base_probability  dapt_alt_rank  dapt_alt_tokenid dapt_alt_token  dapt_alt_probability
0          1           584     ' other'          0.042839              1               511       ' their'              0.023131
1          2           511     ' their'          0.037026              2               262         ' the'              0.011090
2          3          6844      ' dogs'          0.036724              3             10693        ' mice'              0.008588
3          4         11875      ' cats'          0.027796              4               284          ' to'              0.007950
4          5           257         ' a'          0.024118              5               257           ' a'              0.005658
5          6           262       ' the'          0.023013              6             28837       ' lymph'              0.005629
6          7          1751  ' children'          

## 7. Full validation-set loss evaluation across all seven models

In [16]:
from llm_from_scratch.dataloader import dataloader

VAL_ABSTRACTS_PATH = DATA_DIR / "pubmed_abstracts_2005to2025ONLY_ERBB2_ABSTRACTS_getv7_english_val_abstracts.txt"

if not VAL_ABSTRACTS_PATH.exists():
    raise FileNotFoundError(f"Validation abstracts file not found: {VAL_ABSTRACTS_PATH}")

validation_set_fraction = 0.025

if not (0 < validation_set_fraction <= 1):
    raise ValueError(f"validation_set_fraction must be > 0 and <= 1, got {validation_set_fraction}")

val_text_full = dataloader.load_file(VAL_ABSTRACTS_PATH)
val_text_num_chars = max(1, int(len(val_text_full) * validation_set_fraction))
val_text = val_text_full[:val_text_num_chars]

# Match the full-set evaluation style used at the end of training: no shuffle, no dropped last batch,
# and loss computed across the entire validation loader.
val_batch_size = int(checkpoint.get("run_config", {}).get("batch_size", 2))
val_stride = int(checkpoint.get("run_config", {}).get("stride", dapt_model_cfg["context_length"]))
val_context_length = int(dapt_model_cfg["context_length"])

full_val_loader = dataloader.create_dataloader_v1(
    val_text,
    batch_size=val_batch_size,
    max_length=val_context_length,
    stride=val_stride,
    shuffle=False,
    drop_last=True,
    num_workers=0,
)

print("Validation file:", VAL_ABSTRACTS_PATH)
print("Validation set fraction:", validation_set_fraction)
print("Validation characters used:", len(val_text), "of", len(val_text_full))
print("Validation batch size:", val_batch_size)
print("Validation context length:", val_context_length)
print("Validation stride:", val_stride)
print("Number of validation batches:", len(full_val_loader))

def eval_full_val_set(model, val_loader, device):
    was_training = model.training
    model.eval()
    with torch.no_grad():
        val_loss = training_utils.calc_loss_loader(val_loader, model, device, num_batches=None)
    if was_training:
        model.train()
    return val_loss

val_loss_rows = []
models_to_eval = [
    ("base_model", base_model),
    ("base_embed_plusothers_dapt_model", base_embed_plusothers_dapt_model),
    ("base_embed_pos_trf0_11_model", base_embed_pos_trf0_11_model),
    ("base_embed_pos_trf5_6_model", base_embed_pos_trf5_6_model),
    ("base_embed_dapt_model", base_embed_dapt_model),
    ("dapt_model", dapt_model),
    ("dapt_embed_base_model", dapt_embed_base_model),
]

for model_name, model in models_to_eval:
    full_val_loss = eval_full_val_set(model, full_val_loader, device)
    print(f"{model_name} FULL val loss: {full_val_loss:.6f}")
    val_loss_rows.append({
        "model": model_name,
        "validation_set_fraction": validation_set_fraction,
        "full_val_loss": full_val_loss,
    })

val_loss_df = pd.DataFrame(val_loss_rows).sort_values("full_val_loss").reset_index(drop=True)
print()
print(val_loss_df.to_string(index=False))


Validation file: /content/drive/MyDrive/llm-from-scratch-drive/data/pubmed_abstracts_2005to2025ONLY_ERBB2_ABSTRACTS_getv7_english_val_abstracts.txt
Validation set fraction: 0.025
Validation characters used: 72602 of 2904096
Validation batch size: 2
Validation context length: 1024
Validation stride: 1024
Number of validation batches: 8
base_model FULL val loss: 2.958093
base_embed_plusothers_dapt_model FULL val loss: 3.267536
base_embed_pos_trf0_11_model FULL val loss: 2.864814
base_embed_pos_trf5_6_model FULL val loss: 3.194030
base_embed_dapt_model FULL val loss: 3.309281
dapt_model FULL val loss: 2.585541
dapt_embed_base_model FULL val loss: 2.905701

                           model  validation_set_fraction  full_val_loss
                      dapt_model                    0.025       2.585541
    base_embed_pos_trf0_11_model                    0.025       2.864814
           dapt_embed_base_model                    0.025       2.905701
                      base_model              